# Markov Chains

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

A refresher on Markov chains as a procedural-generation tool: model a system as states with
*memoryless* transition probabilities, learn those probabilities from a corpus, then sample
new sequences that statistically resemble it. The workhorse behind text/name generators,
weather-style world events, and music/level grammars.

## 1. What & Why

A **Markov chain** is a model of a system that moves between a finite set of **states**, where
the probability of the next state depends **only on the current state** — not on the full
history of how you got there. That "no memory" assumption is the **Markov property**.

For procedural generation the loop is dead simple:

1. **Learn** a transition table from example data (which states tend to follow which).
2. **Walk** the chain: start somewhere, repeatedly sample the next state from the current
   state's distribution, emit the result.

**The problem it solves.** You want output that *feels like* your source material — fantasy
names that sound Elvish, flavor text in an author's voice, plausible weather sequences, chord
progressions in a genre — without hand-authoring rules. A Markov chain captures local
statistical structure from a handful of examples and lets you generate unlimited variations
cheaply (no GPU, no training loop, milliseconds per sample).

**When to reach for it.** Short, locally-structured sequences where "looks plausible" beats
"globally coherent": name/word generators, simple text, NPC dialogue snippets, procedural
quests/events, drum patterns, tile/room sequences in a dungeon.

**When not to.** Anything needing long-range coherence, grammar, or meaning. A Markov chain has
no concept of sentence structure, plot, or constraints across distant tokens — that's what
context-free grammars, constraint solvers (WFC), or neural language models are for. See
[[context-free-grammars]] and [[wave-function-collapse]].

## 2. Mental Model

Picture a **board game where each square tells you the dice odds for your next move**. You only
ever look at the square you're standing on — never the path that got you there — and roll to pick
where to step next. Play long enough and the *sequence of squares you visit* is your generated
output.

```
        0.5
   ┌──────────────► sunny ──────┐
   │                 ▲  │ 0.3    │ 0.2
 (start)             │  ▼        ▼
   │            0.4  cloudy ──► rainy
   └─────────────────▲   0.6     │ 0.7
                     └───────────┘ 0.3
```

Each node is a state; each arrow is "given I'm here, the chance I go there next." The numbers
on arrows leaving any node sum to 1.0. To generate: drop a token on a node, follow a weighted
random arrow, repeat. The whole model **is** that table of arrow weights — the **transition
matrix** `P`, where `P[i][j] = Pr(next = j | current = i)`.

For text the states are tokens (characters or words). An **order-k** chain just makes the
"current state" the last *k* tokens instead of one — trading memory for local realism.

## 3. Key Concepts

- **State** — one configuration the system can be in (a weather condition, a character, a word,
  the last *k* characters). The set of all states is the **state space**.
- **Markov property** — the next state depends only on the current state. Future ⟂ past, given
  the present. This is the simplifying assumption that makes everything cheap.
- **Transition matrix `P`** — rows = current state, columns = next state, `P[i][j]` = probability.
  Each **row sums to 1**. This object *is* the model.
- **Order (n-gram order)** — how many previous tokens count as the "current state". Order-1 looks
  at 1 token, order-2 at 2, etc. Higher order → more faithful to the source but needs more data
  and starts to **plagiarize** (regurgitate the corpus) as the table approaches one-path-per-key.
- **Sampling / random walk** — generating a sequence by repeatedly drawing the next state from
  the current row's distribution. Use a seeded RNG for reproducibility.
- **Start & stop handling** — you need a way to begin (sample from initial-state frequencies or a
  dedicated `START` token) and to end (an absorbing `STOP`/`END` state, or a length cap).
- **Stationary distribution `π`** — the long-run fraction of time spent in each state, satisfying
  `π = πP`. Tells you the chain's equilibrium behavior; relevant for "what does this converge to."
- **Smoothing** — assigning a little probability to unseen transitions so generation doesn't dead-end
  or memorize. Often skipped in toy generators but matters with sparse data.

## 4. Setup

Pure-Python and NumPy — no special install needed beyond NumPy, which you almost certainly have.
Everything below runs on CPU in milliseconds. The `%pip install` line is included for a bare
environment; it's a no-op if NumPy is already present.

In [ ]:
%pip install -q numpy
import numpy as np
from collections import defaultdict, Counter

print("numpy", np.__version__)

## 5. Worked Examples

Two self-contained examples:

1. **Weather chain** — an explicit transition matrix; sample a sequence and verify the
   empirical stationary distribution matches the theoretical one.
2. **Character-level name generator** — *learn* an order-*k* chain from a small corpus and
   generate new fantasy names. This is the canonical procedural-generation use.

### Example 1 — A weather chain from an explicit matrix

Three states (`sunny`, `cloudy`, `rainy`) with a hand-written transition matrix. We do a random
walk to generate a weather sequence, then check that the fraction of days spent in each state
converges to the **stationary distribution** `π` (the solution of `π = πP`).

In [ ]:
states = ["sunny", "cloudy", "rainy"]
# Rows = today, columns = tomorrow. Each row sums to 1.
P = np.array([
    [0.6, 0.3, 0.1],   # after sunny
    [0.3, 0.4, 0.3],   # after cloudy
    [0.2, 0.4, 0.4],   # after rainy
])
assert np.allclose(P.sum(axis=1), 1.0)  # sanity: valid probability rows

rng = np.random.default_rng(7)

def walk(P, n_steps, start=0):
    s = start
    seq = [s]
    for _ in range(n_steps - 1):
        s = rng.choice(len(P), p=P[s])   # next state ~ current row
        seq.append(s)
    return seq

seq = walk(P, 20, start=0)
print("20-day forecast:")
print(" -> ".join(states[s] for s in seq))

In [ ]:
# Empirical distribution from a long walk vs. theoretical stationary distribution pi = pi P.
long = walk(P, 50_000, start=0)
empirical = np.bincount(long, minlength=3) / len(long)

# Stationary pi is the left eigenvector of P with eigenvalue 1, normalized to sum to 1.
vals, vecs = np.linalg.eig(P.T)
pi = np.real(vecs[:, np.argmin(np.abs(vals - 1))])
pi = pi / pi.sum()

print(f"{'state':8} {'empirical':>10} {'stationary':>11}")
for i, s in enumerate(states):
    print(f"{s:8} {empirical[i]:>10.3f} {pi[i]:>11.3f}")
print("\nMatch:", np.allclose(empirical, pi, atol=0.01))

### Example 2 — Learn an order-*k* character chain and generate names

Now the procedural-generation pattern: **learn** the transition table from data instead of
writing it by hand. We build an **order-*k*** character model over a tiny corpus of fantasy
names — the "state" is the last *k* characters, and we predict the next character. `^` marks the
start padding and `$` marks end-of-name (an absorbing stop state).

In [ ]:
corpus = [
    "aelin", "thalia", "eldrin", "lyra", "kael", "seraphina", "rowan",
    "elowen", "caelum", "isolde", "branwen", "faelar", "maelis", "orin",
    "tamsin", "vaela", "ardyn", "elara", "fenwick", "galad", "lirien",
]

K = 2  # order: condition on the last K characters
START, STOP = "^", "$"

def train(corpus, k):
    model = defaultdict(Counter)
    for word in corpus:
        padded = START * k + word + STOP
        for i in range(len(padded) - k):
            key = padded[i:i + k]        # last k chars
            nxt = padded[i + k]          # the char that follows
            model[key][nxt] += 1
    return model

model = train(corpus, K)
# Peek at what follows the prefix "^^" (name starts) and "el".
print('after "^^":', dict(model[START * K]))
print('after "el":', dict(model["el"]))

In [ ]:
def generate(model, k, max_len=15):
    key = START * k
    out = []
    while len(out) < max_len:
        counter = model.get(key)
        if not counter:           # unseen context -> stop gracefully
            break
        choices = list(counter.keys())
        weights = np.array(list(counter.values()), dtype=float)
        weights /= weights.sum()
        nxt = rng.choice(choices, p=weights)
        if nxt == STOP:
            break
        out.append(nxt)
        key = (key + nxt)[-k:]    # slide the k-char window
    return "".join(out)

print(f"Generated names (order-{K}):")
for _ in range(12):
    name = generate(model, K)
    if name:
        print("  ", name.capitalize())

Tweak `K` to feel the trade-off: `K=1` gives mush that barely resembles names; `K=3` produces
very name-like output but increasingly just **echoes the corpus** because each 3-char context
often has only one observed continuation. Order-2 is the usual sweet spot for short words.

## 6. Gotchas & Pitfalls

- **Higher order = plagiarism, not magic.** As order grows, each key tends to map to a single
  observed next token, so the chain regurgitates training sequences verbatim. More order needs
  *exponentially* more data to stay novel. Pick the smallest order that "sounds right."
- **Dead ends / unseen contexts.** A context that never appeared in training has no row to sample
  from. Handle it: fall back to a lower order (**backoff**), add **smoothing**, or just stop.
  Forgetting this throws `KeyError` or silently truncates output.
- **No global coherence — by design.** The Markov property means the model can't enforce length,
  balance parentheses, avoid repeats, or stay on topic. Don't expect grammatical sentences or
  valid code; constrain or post-process instead.
- **Rows must be normalized.** If you store raw counts, remember to divide by the row sum before
  sampling. Off-by-one normalization bugs skew the output subtly.
- **Start/stop is part of the model.** Sampling start states uniformly (instead of from real
  initial-token frequencies) makes everything look wrong. Use explicit `START`/`STOP` padding.
- **Reproducibility.** Generation is stochastic — seed your RNG (`np.random.default_rng(seed)`)
  when you need deterministic output for tests or save-files.
- **Sparse data → degenerate chains.** With too few examples, the matrix is mostly zeros and the
  chain collapses into a couple of memorized paths. Markov chains are data-hungry relative to
  their simplicity.

## 7. When to Use vs Alternatives

**Reach for a Markov chain when:** you have example sequences, want cheap CPU-only generation,
care about *local* plausibility, and "statistically similar" is good enough. Names, flavor text,
chiptune melodies, weather/event systems, simple dungeon room ordering. Tiny code, instant,
trivially tweakable.

| Approach | Strength | Weakness vs. Markov | Use when |
|---|---|---|---|
| **Markov chain** | Trivial to train & sample; captures local style | No long-range structure or constraints | Short, locally-structured sequences from examples |
| **Context-free grammar** ([[context-free-grammars]]) | Enforces nested structure & rules | You must author rules by hand; no statistics from data | Output must obey a grammar (quests, valid syntax) |
| **Wave Function Collapse** ([[wave-function-collapse]]) | Hard local constraints over 2D/3D, coherent tilemaps | Heavier; can contradict & backtrack | Tile maps where neighbors must be compatible |
| **n-gram + smoothing (NLP)** | Same idea, principled probabilities for real text | Still no deep semantics | Larger-scale text modeling, language ID |
| **Neural LM (RNN/Transformer)** | Long-range coherence, semantics, huge expressivity | Needs data, training, GPU, ms→s latency | Coherent prose/code, meaning matters |
| **Hand-authored templates** | Total control, always valid | No variety; tedious | Few outputs, exact wording required |

Rule of thumb: **Markov for vibe, grammar for structure, neural for meaning.** They also compose —
e.g. a grammar whose terminals are filled by a Markov name generator.

## 8. Resources

- **Setzer / "Markov chains" — Wikipedia** — solid formal grounding (states, transition matrix,
  stationary distribution): <https://en.wikipedia.org/wiki/Markov_chain>
- **"The Unreasonable Effectiveness of Markov Chains" (and classic name-generator write-ups)** —
  procedural-generation framing on RogueBasin: <http://www.roguebasin.com/index.php/Markov_chains>
- **Jay Alammar — Visual intro to text generation with Markov chains / n-grams** —
  intuition-building diagrams: <https://jalammar.github.io/a-visual-and-interactive-guide-to-the-basics-of-neural-networks/>
- **NumPy `random.Generator.choice` docs** — the sampling primitive used above:
  <https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html>
- **"Markov Chains explained visually" — Victor Powell (setosa.io)** — interactive transition
  diagrams you can poke: <https://setosa.io/ev/markov-chains/>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
START = "\x02"
STOP = "\x03"


def train(sequences, order):
    ...


def walk(table, order, pick, max_length=100):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE